# <div style="text-align:center; border-radius:15px 50px; padding:12px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color: #000080; overflow:hidden"><b>Part 2 continued: the Pandas analysis</b><div style="font-family:Arial; font-size:58%; color:#d8dcff; margin-top:8px">presented by Aaska</div></div>

### Module 6: Pandas + NumPy Analysis

**Why we switch to Pandas now.** Reading 2000 lines by hand in Module 5 worked, but the real file
has 52,704 rows across six cities -- not something we want to loop over by hand. Pandas loads the
whole thing at once and gives us `shape`, `groupby`, and everything else we need for the real
analysis. Once the DataFrame is built, Anupam continues this same module with NumPy before we move
on to visualization in Module 7.


In [1]:
import pandas as pd


def load_air_quality_data(path):
    """Load the air quality CSV into a DataFrame."""
    return pd.read_csv(path)


df = load_air_quality_data("../Air_Quality.csv")
df.head()


,Date,City,CO,CO2,NO2,SO2,O3,PM2.5,PM10,AQI
0,2024-01-01 00:00:00+00:00,Brasilia,323.0,NaN,23.8,2.8,42.0,12.0,17.1,16.800000
1,2024-01-01 01:00:00+00:00,Brasilia,318.0,NaN,21.9,2.7,40.0,12.5,17.9,16.000000
2,2024-01-01 02:00:00+00:00,Brasilia,309.0,NaN,19.2,2.6,39.0,12.1,17.3,15.599999
3,2024-01-01 03:00:00+00:00,Brasilia,295.0,NaN,16.3,2.4,38.0,11.4,16.2,15.200000
4,2024-01-01 04:00:00+00:00,Brasilia,270.0,NaN,13.0,2.1,40.0,10.2,14.6,16.000000


In [2]:
def describe_dataframe(df):
    """Print the shape, columns, and dtypes of a DataFrame."""
    print("Shape (rows, columns):", df.shape)
    print()
    print("Columns:", list(df.columns))
    print()
    print("Data types:")
    print(df.dtypes)


describe_dataframe(df)


Shape (rows, columns): (52704, 10)

Columns: ['Date', 'City', 'CO', 'CO2', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']

Data types:
Date         str
City         str
CO       float64
CO2      float64
NO2      float64
SO2      float64
O3       float64
PM2.5    float64
PM10     float64
AQI      float64
dtype: object


**Checking for missing values** across every column.

In [3]:
def summarize_missing_values(df):
    """Return a DataFrame of missing counts and percentages per column."""
    missing_counts = df.isnull().sum()
    missing_percent = (missing_counts / len(df) * 100).round(2)
    return pd.DataFrame({
        "missing_count": missing_counts,
        "missing_percent": missing_percent,
    })


missing_summary = summarize_missing_values(df)
missing_summary


,missing_count,missing_percent
Date,0,0.00
City,0,0.00
CO,0,0.00
CO2,43056,81.69
NO2,0,0.00
SO2,0,0.00
O3,0,0.00
PM2.5,0,0.00
PM10,0,0.00
AQI,0,0.00


**What this shows.** `CO2` is the only column with anything missing, and it is missing in 81.7% of
all rows -- far too much to just fill in and move on. Before deciding what to do about it, we check
*when* the missing values happen, instead of assuming they are scattered randomly.


In [4]:
def add_month_column(df):
    """Parse the Date column and add a Month column, returning the updated DataFrame."""
    df["Date"] = pd.to_datetime(df["Date"])
    df["Month"] = df["Date"].dt.month
    return df


def co2_availability_pattern(df, city):
    """Return CO2 readings-present vs readings-total by month for one city."""
    city_df = df[df["City"] == city]
    readings_present = city_df.groupby("Month")["CO2"].apply(lambda col: col.notnull().sum())
    readings_total = city_df.groupby("Month")["CO2"].size()
    return pd.DataFrame({
        "readings_present": readings_present,
        "readings_total": readings_total,
    })


df = add_month_column(df)
co2_pattern = co2_availability_pattern(df, "Brasilia")
co2_pattern


,readings_present,readings_total
Month,,
1,0,744
2,0,696
3,0,744
4,0,720
5,0,744
6,0,720
7,0,744
8,0,744
9,0,720


**What we found.** CO2 has zero readings from January through most of October, then becomes
essentially complete from late October onward -- a monitoring change that started on **26 October
2024**, the same date across every single city. This is a **structural gap**, not a handful of
random sensor dropouts, so filling it in with 0, the column average, or interpolation across the
whole year would fabricate ten months of data that were never actually measured.

**Our decision.** We do not invent CO2 values for January through October. Instead:
- CO2 is only reported and used for the period where it was genuinely measured (26 Oct 2024 onward).
- Every other pollutant (CO, NO2, SO2, O3, PM2.5, PM10) and AQI is complete for the full year, and
  those are what drive the city/time analysis below.


In [5]:
def summarize_co2_availability(df):
    """Print CO2 coverage stats and return average CO2 by city where available."""
    co2_available = df[df["CO2"].notnull()]
    print("Earliest CO2 reading :", co2_available["Date"].min())
    print("Latest CO2 reading   :", co2_available["Date"].max())
    print("Rows with usable CO2 :", len(co2_available), f"({len(co2_available)/len(df)*100:.1f}% of all rows)")
    print()
    print("Average CO2 (Oct 26 - Dec 31 only), by city:")
    return co2_available.groupby("City")["CO2"].mean().round(1)


summarize_co2_availability(df)


Earliest CO2 reading : 2024-10-26 00:00:00+00:00
Latest CO2 reading   : 2024-12-31 23:00:00+00:00
Rows with usable CO2 : 9648 (18.3% of all rows)

Average CO2 (Oct 26 - Dec 31 only), by city:


City
Brasilia    445.7
Cairo       457.9
Dubai       463.8
London      475.1
New York    488.4
Sydney      443.2
Name: CO2, dtype: float64

**One more thing worth noticing.** In this two-month window New York has the highest average CO2
and Sydney the lowest -- a different ranking than the AQI ranking we get below for the full year.
That is one more reason not to stretch CO2 conclusions over months we never actually measured.


**Filtering.** Every hour where AQI reaches at least "Unhealthy for Sensitive Groups" (AQI > 100).

In [6]:
def top_high_aqi_hours(df, threshold=100, top_n=10):
    """Print how many hours exceed the AQI threshold and return the worst top_n rows."""
    high_aqi_df = df[df["AQI"] > threshold]
    print(f"Hours with AQI > {threshold}:", len(high_aqi_df), f"out of {len(df)} ({len(high_aqi_df)/len(df)*100:.2f}%)")
    return high_aqi_df[["Date", "City", "AQI", "PM2.5", "PM10"]].sort_values("AQI", ascending=False).head(top_n)


top_high_aqi_hours(df)


Hours with AQI > 100: 2182 out of 52704 (4.14%)


,Date,City,AQI,PM2.5,PM10
23092,2024-08-18 04:00:00+00:00,Dubai,196.63333,65.9,391.2
23091,2024-08-18 03:00:00+00:00,Dubai,196.41666,59.3,339.0
23093,2024-08-18 05:00:00+00:00,Dubai,195.93500,70.8,443.3
23090,2024-08-18 02:00:00+00:00,Dubai,195.22168,53.0,314.8
23094,2024-08-18 06:00:00+00:00,Dubai,195.19000,69.6,452.6
23103,2024-08-18 15:00:00+00:00,Dubai,194.91167,52.4,271.0
23104,2024-08-18 16:00:00+00:00,Dubai,194.89668,53.4,278.5
23095,2024-08-18 07:00:00+00:00,Dubai,194.80164,64.7,431.2
23096,2024-08-18 08:00:00+00:00,Dubai,194.41333,60.1,412.6
23102,2024-08-18 14:00:00+00:00,Dubai,194.27502,52.6,276.6


**What jumps out.** Every single one of the top 10 worst hours is Dubai on 18 August -- the same
event we picked out by hand in Aishmita's Part 1. It is not spread across the year; it is one event.


**Grouping by city.** Average pollutant levels and AQI, city by city.

In [7]:
def summarize_by_city(df):
    """Return average pollutant levels and AQI by city, sorted worst to best."""
    city_summary = df.groupby("City")[["CO", "NO2", "SO2", "O3", "PM2.5", "PM10", "AQI"]].mean().round(2)
    return city_summary.sort_values("AQI", ascending=False)


city_summary = summarize_by_city(df)
city_summary


,CO,NO2,SO2,O3,PM2.5,PM10,AQI
City,,,,,,,
Dubai,440.46,39.49,20.31,93.88,41.57,111.04,85.11
Cairo,293.82,32.62,38.29,61.10,23.74,43.42,52.49
New York,283.64,27.80,6.25,52.12,13.36,18.95,32.88
London,187.74,21.80,3.33,49.15,9.99,14.14,27.10
Sydney,122.43,14.17,5.62,46.14,10.94,17.42,25.77
Brasilia,221.45,8.74,1.62,57.76,6.55,8.89,24.75


**Date/time analysis.** Average AQI by month, across all six cities.

In [8]:
def summarize_by_month(df):
    """Return average AQI by month across all cities."""
    return df.groupby("Month")["AQI"].mean().round(2)


monthly_aqi = summarize_by_month(df)
monthly_aqi


Month
1     39.45
2     39.24
3     41.76
4     40.83
5     40.24
6     40.66
7     46.81
8     48.33
9     43.84
10    35.85
11    38.21
12    40.78
Name: AQI, dtype: float64

**Handing off to Anupam.** `city_summary` and `monthly_aqi` are the two tables Anupam turns into
charts in Part 3, and the arrays below feed straight into the NumPy half of Module 6.


# <div style="text-align:center; border-radius:15px 50px; padding:12px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color: #191970; overflow:hidden"><b>Part 3: NumPy and visualization</b><div style="font-family:Arial; font-size:58%; color:#d8dcff; margin-top:8px">presented by Anupam</div></div>

**Module 6 continued -- NumPy analysis.** Picking up from Aaska's DataFrame, `df` already has
everything we need, so we pull the key pollutant columns out as NumPy arrays and compute summary
statistics and comparisons directly with NumPy instead of Pandas' built-in methods.


In [9]:
import numpy as np


def extract_pollutant_arrays(df):
    """Convert key pollutant columns to NumPy arrays."""
    return {
        "AQI": df["AQI"].to_numpy(),
        "PM2.5": df["PM2.5"].to_numpy(),
        "PM10": df["PM10"].to_numpy(),
        "CO": df["CO"].to_numpy(),
        "NO2": df["NO2"].to_numpy(),
        "SO2": df["SO2"].to_numpy(),
        "O3": df["O3"].to_numpy(),
    }


pollutant_arrays = extract_pollutant_arrays(df)
aqi_array = pollutant_arrays["AQI"]
pm25_array = pollutant_arrays["PM2.5"]
pm10_array = pollutant_arrays["PM10"]
co_array = pollutant_arrays["CO"]
no2_array = pollutant_arrays["NO2"]
so2_array = pollutant_arrays["SO2"]
o3_array = pollutant_arrays["O3"]

print("AQI array shape:", aqi_array.shape, " dtype:", aqi_array.dtype)


AQI array shape: (52704,)  dtype: float64


In [10]:
def print_array_stats(pollutant_arrays):
    """Print mean/min/max/std for each pollutant array."""
    for name, arr in pollutant_arrays.items():
        print(f"{name:6s}  mean={np.mean(arr):8.2f}  min={np.min(arr):8.2f}  max={np.max(arr):8.2f}  std={np.std(arr):8.2f}")


print_array_stats(pollutant_arrays)


AQI     mean=   41.35  min=    4.45  max=  196.63  std=   26.63
PM2.5   mean=   17.69  min=    0.10  max=  129.50  std=   15.67
PM10    mean=   35.64  min=    0.10  max=  543.90  std=   48.44
CO      mean=  258.26  min=   52.00  max= 2045.00  std=  159.59
NO2     mean=   24.10  min=    0.00  max=  165.90  std=   19.36
SO2     mean=   12.57  min=    0.00  max=  239.70  std=   17.27
O3      mean=   60.03  min=    0.00  max=  349.00  std=   38.22


**Numerical comparisons.** Counting how many hourly readings cross common pollution thresholds.

In [11]:
def count_threshold_exceedances(aqi_array, pm25_array, pm10_array):
    """Count hours exceeding common pollutant thresholds."""
    unhealthy_aqi_hours = np.sum(aqi_array > 100)
    high_pm25_hours = np.sum(pm25_array > 35)
    high_pm10_hours = np.sum(pm10_array > 150)
    return unhealthy_aqi_hours, high_pm25_hours, high_pm10_hours


unhealthy_aqi_hours, high_pm25_hours, high_pm10_hours = count_threshold_exceedances(aqi_array, pm25_array, pm10_array)

print("Hours with AQI > 100   :", unhealthy_aqi_hours)
print("Hours with PM2.5 > 35  :", high_pm25_hours)
print("Hours with PM10 > 150  :", high_pm10_hours)


Hours with AQI > 100   : 2182
Hours with PM2.5 > 35  : 6655
Hours with PM10 > 150  : 2110


**Correlation.** How closely PM2.5 and PM10 track AQI.

In [12]:
def compute_correlations(pm25_array, pm10_array, aqi_array):
    """Return the correlation of PM2.5 and PM10 with AQI."""
    pm25_aqi_corr = np.corrcoef(pm25_array, aqi_array)[0, 1]
    pm10_aqi_corr = np.corrcoef(pm10_array, aqi_array)[0, 1]
    return pm25_aqi_corr, pm10_aqi_corr


pm25_aqi_corr, pm10_aqi_corr = compute_correlations(pm25_array, pm10_array, aqi_array)

print(f"Correlation between PM2.5 and AQI: {pm25_aqi_corr:.3f}")
print(f"Correlation between PM10 and AQI : {pm10_aqi_corr:.3f}")


Correlation between PM2.5 and AQI: 0.822
Correlation between PM10 and AQI : 0.846


**What this means.** Both correlations are strongly positive, which fits, since AQI is partly
*calculated from* particulate readings in the first place -- this confirms the relationship rather
than discovering a surprising one. We treat it as a confirmed pattern, not a causal claim.
